# Nettoyage de la base d'apprentissage

In [69]:
import numpy as np
import pandas as pd
import sys
import os

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from cleaning import *

# On charge la base d'apprentissage
df = pd.read_csv("../data_finale/base_apprentissage.csv")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Traitement des doublons

In [70]:
verifier_doublons_metier_et_techniques(df)

Recherche de doublons
 ATTENTION : 787 lignes sont des doublons techniques stricts.
   (Même joueur, même saison, même club -> Erreur d'extraction/jointure)

   Exemple de lignes techniques concernées :
                player  season         team
28             Willian    2021      Arsenal
29             Willian    2021      Arsenal
30             Willian    2021      Arsenal
37   Emiliano Martínez    2021  Aston Villa
38   Emiliano Martínez    2021  Aston Villa
119           Jorginho    2021      Chelsea
=869 lignes correspondent à des doublons de mercato
   (Même joueur, même saison, mais CLUBS DIFFÉRENTS -> Transferts de mi-saison)

   Exemple de joueurs transférés concernés :
                    player  season     team
0   Ainsley Maitland-Niles    2021  Arsenal
14             Joe Willock    2021  Arsenal
16         Martin Ødegaard    2021  Arsenal
17             Mathew Ryan    2021  Arsenal
25          Sead Kolašinac    2021  Arsenal
26        Shkodran Mustafi    2021  Arsenal


{'doublons_techniques': np.int64(787), 'doublons_mercato': np.int64(869)}

In [71]:
df = fusionner_doublons_techniques(df)

Format initial de la base : (17122, 134)
Format après fusion intelligente des doublons : (16335, 134)


In [72]:
verifier_doublons_mercato(df)

Diagnostic des doublons liés au mercato (transferts de mi-saison)
Attention : 869 lignes sont des doublons pour le même joueur lors de la même saison !
   Cela indique la présence de transferts ou de prêts à la mi-saison.

   Exemple de lignes concernées :
             player  season        team
40     Aarón Martín    2021  Celta Vigo
41     Aarón Martín    2021    Mainz 05
49     Abakar Sylla    2526      Nantes
50     Abakar Sylla    2526  Strasbourg
58  Abde Ezzalzouli    2324   Barcelona
59  Abde Ezzalzouli    2324  Real Betis


In [73]:
df = fusionner_et_recalculer_mercato(df)

Format avant fusion mercato : (16335, 134)


Format après fusion mercato : (15466, 134)
Recalcul des ratios et statistiques par 90 minutes...
Base de données fusionnée et variables recalculées avec exactitude.



<string>:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<string>:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


In [74]:
df

,player,season,team,league,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,injury_minor_unknown_nb_d,injury_minor_unknown_nb_m,injury_musculaire,injury_genou,injury_cheville_pied,injury_mollet_tibia,injury_dos_bassin,injury_trauma_severe,injury_medical_repos,injury_minor_unknown
0,Aaron Ciammaglichella,2425,Torino,ITA-Serie A,ITA,MF,19,2005.0,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Aaron Connolly,2021,Brighton,ENG-Premier League,IRL,"FW,MF",20,2000.0,17,9,...,29.0,6.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,Aaron Connolly,2122,Brighton,ENG-Premier League,IRL,FW,21,2000.0,4,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Aaron Cresswell,2021,West Ham United,ENG-Premier League,ENG,DF,30,1989.0,36,36,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,Aaron Cresswell,2122,West Ham United,ENG-Premier League,ENG,DF,31,1989.0,31,31,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15461,Šime Vrsaljko,2021,Atlético Madrid,ESP-La Liga,CRO,MF,28,1992.0,9,6,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
15462,Šime Vrsaljko,2122,Atlético Madrid,ESP-La Liga,CRO,"DF,MF",29,1992.0,21,10,...,193.0,10.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
15463,Ștefan Radu,2021,Lazio,ITA-Serie A,ROU,DF,33,1986.0,31,30,...,24.0,5.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
15464,Ștefan Radu,2122,Lazio,ITA-Serie A,ROU,DF,34,1986.0,10,6,...,36.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


In [75]:
# A FAIRE APRES LA GESTION DES DOUBLONS MERCATOS

# df = df[df["Playing Time_Min"] > (5 * 90)]

## Traitement des variables avec beaucoup de valeurs manquantes

In [76]:
diagnostiquer_valeurs_manquantes(df, seuil=0.30)

Diagnostic des valeurs manquantes (Seuil > 30%)
   • Aucune colonne ne dépasse 30% de lignes vides.


[]

## Encodage de variables

In [77]:
# Analyser la base des gardiens
var_categorielles = lister_variables_categorielles(df)

Variables catégorielles du dataset :
Liste des variables catégorielles détectées :
   • player (5511 modalités uniques)
   • team (137 modalités uniques)
   • league (5 modalités uniques)
   • nation (130 modalités uniques)
   • pos (10 modalités uniques)
   • age (1757 modalités uniques)
   • join_key (5509 modalités uniques)
   • match_method (17 modalités uniques)
   • date (249 modalités uniques)
   • date_of_birth (3270 modalités uniques)
   • name (4548 modalités uniques)
   • tm_join_key (4547 modalités uniques)
   • tm_join_key_full (4547 modalités uniques)
   • sub_position (14 modalités uniques)
   • position (6 modalités uniques)
   • foot (4 modalités uniques)
   • contract_expiration_date (40 modalités uniques)

Total : 17 variables catégorielles trouvées.


In [78]:
# Traitement de la base
df_pret = encoder_dataset_football(df)

Format initial avant encodage : (15466, 134)
Profil Joueurs de champ détecté : Encodage Multi-Label des 10 modalités de postes.
   • Encodage des postes terminé. Classes détectées : ['DF', 'FW', 'GK', 'MF']
Colonne 'nation' regroupée : Top 10 + 'Autre' (11 modalités au total).
Format final après encodage : (15466, 151)



In [79]:
df_pret

,player,season,team,age,born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,...,nation_group_Autre,nation_group_BEL,nation_group_BRA,nation_group_ENG,nation_group_ESP,nation_group_FRA,nation_group_GER,nation_group_ITA,nation_group_NED,nation_group_POR
0,Aaron Ciammaglichella,2425,Torino,19,2005.0,1,0,1,0.0,0,...,0,0,0,0,0,0,0,1,0,0
1,Aaron Connolly,2021,Brighton,20,2000.0,17,9,791,8.8,2,...,1,0,0,0,0,0,0,0,0,0
2,Aaron Connolly,2122,Brighton,21,2000.0,4,1,156,1.7,0,...,1,0,0,0,0,0,0,0,0,0
3,Aaron Cresswell,2021,West Ham United,30,1989.0,36,36,3170,35.2,0,...,0,0,0,1,0,0,0,0,0,0
4,Aaron Cresswell,2122,West Ham United,31,1989.0,31,31,2726,30.3,2,...,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15461,Šime Vrsaljko,2021,Atlético Madrid,28,1992.0,9,6,519,5.8,0,...,1,0,0,0,0,0,0,0,0,0
15462,Šime Vrsaljko,2122,Atlético Madrid,29,1992.0,21,10,914,10.2,1,...,1,0,0,0,0,0,0,0,0,0
15463,Ștefan Radu,2021,Lazio,33,1986.0,31,30,2458,27.3,0,...,1,0,0,0,0,0,0,0,0,0
15464,Ștefan Radu,2122,Lazio,34,1986.0,10,6,556,6.2,0,...,1,0,0,0,0,0,0,0,0,0


## Sauvegarde des bases d'apprentissage mises à jour

In [80]:
df_pret.to_csv(r'..\data_finale\base_apprentissage_maj.csv', index=False, sep=',', encoding='utf-8-sig')